# 01. TTS 엔진 비교 (박세준)

| 엔진 | 종류 | 비고 |
|---|---|---|
| MeloTTS | 로컬 모델 (VITS 계열), MIT | 설치가 까다로움 |
| MMS-TTS (facebook/mms-tts-kor) | 로컬 모델 (VITS), CC-BY-NC 4.0 | transformers 만 있으면 됨 |
| edge-tts | 온라인 | 인터넷 필요 |
| gTTS | 온라인 | 인터넷 필요 |
| 브라우저 내장 음성 | 기기 OS | 기기마다 목소리가 달라서 표에서는 따로 적는다 |

재는 것
1. 합성 시간(첫 호출은 모델 올리는 시간이 섞여서 따로), 음성 길이, RTF(합성 시간 / 음성 길이)
2. 숫자·날짜를 맞게 읽는지 (사람이 듣고 표시)
3. 자연스러움 5점 평가 (팀원이 엔진 이름을 모르게 듣기)

런타임은 GPU(T4)를 권한다. CPU 로도 돌지만 로컬 모델이 느리다.

In [ ]:
# 레포 받기 (이미 있으면 최신으로) + 기본 설치
import os
REPO = "/content/AI_tts_web"
if not os.path.exists(REPO):
    !git clone -q https://github.com/DL-2026F-Team05/AI_tts_web.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt -r requirements-dev.txt
!git log --oneline -3

## 엔진 설치
안 되는 엔진은 건너뛰어도 된다. 표에서 빠질 뿐이다.

In [ ]:
!pip install -q -r requirements-tts.txt   # edge-tts, gTTS, uroman(MMS)

In [ ]:
# MeloTTS (선택, 몇 분 걸림)
# 오래된 고정 버전 때문에 최신 코랩에서 설치가 깨질 수 있어서 몇 개만 고정을 푼다.
!git clone -q https://github.com/myshell-ai/MeloTTS.git /content/MeloTTS || git -C /content/MeloTTS pull -q
!sed -i -E 's/^(transformers|tokenizers|librosa|fugashi|numpy|torch|torchaudio)==.*/\1/' /content/MeloTTS/requirements.txt
!pip install -q -e /content/MeloTTS
!python -m unidic download
# 설치 후 런타임을 다시 시작하라고 나오면 다시 시작하고, 위의 레포 받기 셀부터 다시 실행

## 비교 실행

In [ ]:
import time, statistics
from pathlib import Path
import pandas as pd
import librosa
from IPython.display import Audio, display
from packreader.speech.normalize import normalize_for_tts
from packreader.speech.tts import ENGINES

SENTENCES = [
    "주의하세요. 우유가 들어 있어요. 소비기한은 2027년 3월 15일까지예요. 125일 남았어요.",
    "유통기한은 11월 25일까지예요. 15일 남았어요. 제품은 신선 우유예요.",
    "땅콩이 섞였을 수 있어요. 같은 시설에서 함께 만들어요.",
    "영양정보: 총 내용량 60g, 320kcal, 나트륨 180mg.",
    "소비기한이 6월 10일까지예요. 2일 남았어요.",
]
OUT = Path("/content/tts_compare")
OUT.mkdir(exist_ok=True)

rows = []
for name in ["melo", "mms", "edge", "gtts"]:
    eng = ENGINES[name]()
    if not eng.available():
        print(f"{name}: 설치 안 됨, 건너뜀")
        continue
    try:
        t0 = time.perf_counter()
        eng.synthesize(normalize_for_tts("준비 중이에요."))
        cold = time.perf_counter() - t0
    except Exception as e:
        print(f"{name}: 실패 {type(e).__name__}: {e}")
        continue
    for i, text in enumerate(SENTENCES):
        t0 = time.perf_counter()
        clip = eng.synthesize(normalize_for_tts(text))
        sec = time.perf_counter() - t0
        path = OUT / f"{name}_{i}.{clip.ext}"
        path.write_bytes(clip.data)
        dur = librosa.get_duration(path=str(path))
        rows.append({"engine": name, "sentence": i, "synth_sec": round(sec, 3), "audio_sec": round(dur, 2),
                     "rtf": round(sec / dur, 3) if dur else None, "cold_start_sec": round(cold, 2)})
    print(f"{name}: 완료")

df = pd.DataFrame(rows)
summary = df.groupby("engine").agg(synth_median=("synth_sec", "median"), rtf_median=("rtf", "median"),
                                   cold_start=("cold_start_sec", "first")).round(3)
summary

In [ ]:
# 들어 보기
for name in df.engine.unique():
    print("==", name)
    for i in range(len(SENTENCES)):
        files = list(OUT.glob(f"{name}_{i}.*"))
        if files:
            print(SENTENCES[i])
            display(Audio(filename=str(files[0])))

## 블라인드 평가표 만들기
엔진 이름을 A, B, C… 로 가려서 팀원에게 들려준다. 정답표(`key.csv`)는 평가가 끝날 때까지 열지 않는다.

In [ ]:
import random, shutil
random.seed(7)
engines = list(df.engine.unique())
labels = dict(zip(engines, random.sample("ABCDEFG"[: len(engines)], len(engines))))
blind = OUT / "blind"
blind.mkdir(exist_ok=True)
sheet = []
for name, lab in labels.items():
    for f in OUT.glob(f"{name}_*.*"):
        i = f.stem.split("_")[1]
        shutil.copy(f, blind / f"{lab}_{i}{f.suffix}")
        sheet.append({"file": f"{lab}_{i}{f.suffix}", "sentence": SENTENCES[int(i)],
                      "자연스러움(1-5)": "", "숫자·날짜 정확(O/X)": "", "알아듣기(1-5)": ""})
pd.DataFrame(sorted(sheet, key=lambda r: r["file"])).to_csv(OUT / "rating_sheet.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{"label": v, "engine": k} for k, v in labels.items()]).to_csv(OUT / "key.csv", index=False)
print("평가표:", OUT / "rating_sheet.csv", "| 음성:", blind)

In [ ]:
# 결과를 드라이브에 보관 (선택)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/DL-Team05/outputs
!cp -r /content/tts_compare /content/drive/MyDrive/DL-Team05/outputs/
df.to_csv("/content/drive/MyDrive/DL-Team05/outputs/tts_compare/timing.csv", index=False)